In [1]:
!pip install openpyxl




   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# load the dataset
data = pd.read_csv('kenya_county_crime.csv')

# clean column names just in case
data.columns = data.columns.str.strip()

def check_severity(crime):
    crime = str(crime).lower()
    
    violent_crimes = ['murder', 'robbery', 'gbv', 'assault']
    petty_crimes = ['theft', 'burglary', 'pickpocketing', 'fraud']
    
    for c in violent_crimes:
        if c in crime:
            return 'Violent'
            
    for c in petty_crimes:
        if c in crime:
            return 'Petty'
            
    return 'Other'

# add severity column
data['Severity'] = data['Crime_Type'].apply(check_severity)

print("Available Locations:")
counties = data['County'].dropna().unique()

for c in counties:
    # get wards for each county
    wards = data[data['County'] == c]['Ward'].unique()
    ward_list = ", ".join([str(w).strip() for w in wards])
    print(f"- {c}: {ward_list}")

print("-" * 40)

# get user input
loc = input("Enter a county or ward to analyze: ")

# filter by either county or ward
mask = data['County'].str.contains(loc, case=False, na=False) | \
       data['Ward'].str.contains(loc, case=False, na=False)

filtered_data = data[mask]

if len(filtered_data) == 0:
    print("Location not found. Please try again.")
else:
    print(f"\nCrime Analysis for: {loc.upper()}")
    print("-" * 40)

    # get top crimes
    top_crimes = filtered_data.groupby('Crime_Type')['Cases'].sum().sort_values(ascending=False)
    
    print("Top Crimes Reported:")
    for crime, count in top_crimes.head(5).items():
        print(f"* {crime}: {count}")

    # check if it's a red zone or orange zone
    severity_totals = filtered_data.groupby('Severity')['Cases'].sum()
    
    violent_cases = severity_totals.get('Violent', 0)
    petty_cases = severity_totals.get('Petty', 0)

    print("\nZone Status:")
    if violent_cases >= petty_cases and violent_cases > 0:
        print("RED ZONE")
        print(f"Warning: High number of violent crimes ({violent_cases} violent vs {petty_cases} petty).")
    elif petty_cases > violent_cases:
        print("ORANGE ZONE")
        print(f"Mostly property/petty crimes ({petty_cases} petty vs {violent_cases} violent).")
    else:
        print("Unknown Zone")


Available Locations:
- Nairobi: Kilimani, Kibera, Westlands, Eastleigh
- Mombasa: Nyali, Likoni, Kisauni, Tudor
- Kiambu: Ruiru, Thika, Kikuyu, Limuru
- Nakuru: Naivasha, Gilgil, Molo, Rongai
----------------------------------------

Crime Analysis for: NAIROBI
----------------------------------------
Top Crimes Reported:
* Petty Theft: 300
* Pickpocketing: 210
* Fraud: 159
* Robbery with Violence: 153
* Burglary: 120

Zone Status:
ORANGE ZONE
Mostly property/petty crimes (789 petty vs 288 violent).


In [6]:
import pandas as pd

# loading the data
data = pd.read_csv('kenya_county_crime.csv')

# clean column names just in case
data.columns = data.columns.str.strip()

def check_severity(crime):
    crime = str(crime).lower()
    
    violent_crimes = ['murder', 'robbery', 'gbv', 'assault']
    petty_crimes = ['theft', 'burglary', 'pickpocketing', 'fraud']
    
    for c in violent_crimes:
        if c in crime:
            return 'Violent'
            
    for c in petty_crimes:
        if c in crime:
            return 'Petty'
            
    return 'Other'

# add severity column
data['Severity'] = data['Crime_Type'].apply(check_severity)

print("Available Locations:")
counties = data['County'].dropna().unique()

for c in counties:
    # get wards for each county
    wards = data[data['County'] == c]['Ward'].unique()
    ward_list = ", ".join([str(w).strip() for w in wards])
    print(f"- {c}: {ward_list}")

print("\nAvailable Crimes:")
crimes = data['Crime_Type'].dropna().unique()
crime_list = ", ".join([str(cr).strip() for cr in crimes])
print(crime_list)

print("-" * 40)

# get user input
search_term = input("Enter a county, ward, or crime to analyze: ")

# filter by county, ward, OR crime type
mask = data['County'].str.contains(search_term, case=False, na=False) | \
       data['Ward'].str.contains(search_term, case=False, na=False) | \
       data['Crime_Type'].str.contains(search_term, case=False, na=False)

filtered_data = data[mask]

if len(filtered_data) == 0:
    print("No data found for that search. Please try again.")
else:
    print(f"\nAnalysis Results for: {search_term.upper()}")
    print("-" * 40)
    
    # if they searched for a crime, show where it happens most
    # if they searched for a location, show what crimes happen most there
    is_crime_search = any(data['Crime_Type'].str.contains(search_term, case=False, na=False))
    
    if is_crime_search:
        print("Top Locations for this Crime:")
        # group by County and Ward
        top_locations = filtered_data.groupby(['County', 'Ward'])['Cases'].sum().sort_values(ascending=False)
        for loc, count in top_locations.head(5).items():
            print(f"* {loc[0]} ({loc[1]}): {count} cases")
    else:
        print("Top Crimes Reported:")
        top_crimes = filtered_data.groupby('Crime_Type')['Cases'].sum().sort_values(ascending=False)
        for crime, count in top_crimes.head(5).items():
            print(f"* {crime}: {count} cases")

    # overall severity check
    severity_totals = filtered_data.groupby('Severity')['Cases'].sum()
    
    violent_cases = severity_totals.get('Violent', 0)
    petty_cases = severity_totals.get('Petty', 0)

    print("\nZone/Crime Status:")
    if violent_cases >= petty_cases and violent_cases > 0:
        print("RED ZONE / HIGH SEVERITY")
        print(f"Warning: High number of violent records ({violent_cases} violent vs {petty_cases} petty).")
    elif petty_cases > violent_cases:
        print("ORANGE ZONE / LOWER SEVERITY")
        print(f"Mostly property/petty records ({petty_cases} petty vs {violent_cases} violent).")
    else:
        print("Unknown Status")


Available Locations:
- Nairobi: Kilimani, Kibera, Westlands, Eastleigh
- Mombasa: Nyali, Likoni, Kisauni, Tudor
- Kiambu: Ruiru, Thika, Kikuyu, Limuru
- Nakuru: Naivasha, Gilgil, Molo, Rongai

Available Crimes:
Murder, Robbery with Violence, Burglary, Assault, Petty Theft, Fraud, Pickpocketing, GBV, Theft
----------------------------------------

Analysis Results for: GBV
----------------------------------------
Top Locations for this Crime:
* Kiambu (Limuru): 100 cases
* Nakuru (Molo): 45 cases
* Nairobi (Eastleigh): 40 cases
* Mombasa (Tudor): 35 cases
* Kiambu (Kikuyu): 25 cases

Zone/Crime Status:
RED ZONE / HIGH SEVERITY


In [8]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# load the dataset
data = pd.read_csv('kenya_county_crime.csv')

# clean column names just in case
data.columns = data.columns.str.strip()

def check_severity(crime):
    crime = str(crime).lower()
    
    violent_crimes = ['murder', 'robbery', 'gbv', 'assault']
    petty_crimes = ['theft', 'burglary', 'pickpocketing', 'fraud']
    
    for c in violent_crimes:
        if c in crime:
            return 'Violent'
            
    for c in petty_crimes:
        if c in crime:
            return 'Petty'
            
    return 'Other'

# function to determine zone status for a specific location
def get_location_zone(county, ward):
    # get all crimes for this specific ward
    loc_data = data[(data['County'] == county) & (data['Ward'] == ward)]
    severity_totals = loc_data.groupby('Severity')['Cases'].sum()
    
    violent_cases = severity_totals.get('Violent', 0)
    petty_cases = severity_totals.get('Petty', 0)
    
    if violent_cases >= petty_cases and violent_cases > 0:
        return "RED ZONE"
    elif petty_cases > violent_cases:
        return "ORANGE ZONE"
    else:
        return "UNKNOWN"

# add severity column
data['Severity'] = data['Crime_Type'].apply(check_severity)

print("Available Locations:")
counties = data['County'].dropna().unique()

for c in counties:
    wards = data[data['County'] == c]['Ward'].unique()
    ward_list = ", ".join([str(w).strip() for w in wards])
    print(f"- {c}: {ward_list}")

print("\nAvailable Crimes:")
crimes = data['Crime_Type'].dropna().unique()
crime_list = ", ".join([str(cr).strip() for cr in crimes])
print(crime_list)

print("-" * 40)

# get user input
search_term = input("Enter a county, ward, or crime to analyze: ").strip()

# filter by county, ward, OR crime type
mask = data['County'].str.contains(search_term, case=False, na=False) | \
       data['Ward'].str.contains(search_term, case=False, na=False) | \
       data['Crime_Type'].str.contains(search_term, case=False, na=False)

filtered_data = data[mask]

if len(filtered_data) == 0:
    print("No data found for that search. Please try again.")
else:
    print(f"\nAnalysis Results for: {search_term.upper()}")
    print("-" * 40)
    
    # check if the user searched for a crime instead of a location
    is_crime_search = any(data['Crime_Type'].str.contains(search_term, case=False, na=False))
    
    if is_crime_search:
        print("Top Locations for this Crime:")
        # group by County and Ward
        top_locations = filtered_data.groupby(['County', 'Ward'])['Cases'].sum().sort_values(ascending=False)
        
        for loc, count in top_locations.head(5).items():
            county_name = loc[0]
            ward_name = loc[1]
            
            # evaluate the zone for this specific location
            zone_status = get_location_zone(county_name, ward_name)
            
            print(f"* {county_name} ({ward_name}): {count} cases [{zone_status}]")
            
    else:
        print("Top Crimes Reported:")
        top_crimes = filtered_data.groupby('Crime_Type')['Cases'].sum().sort_values(ascending=False)
        for crime, count in top_crimes.head(5).items():
            print(f"* {crime}: {count} cases")

    # overall severity check for the search results
    severity_totals = filtered_data.groupby('Severity')['Cases'].sum()
    violent_cases = severity_totals.get('Violent', 0)
    petty_cases = severity_totals.get('Petty', 0)

    print("\nOverall Status for Search:")
    if violent_cases >= petty_cases and violent_cases > 0:
        print("RED ZONE / HIGH SEVERITY")
        print(f"Warning: High number of violent records ({violent_cases} violent vs {petty_cases} petty).")
    elif petty_cases > violent_cases:
        print("ORANGE ZONE / LOWER SEVERITY")
        print(f"Mostly property/petty records ({petty_cases} petty vs {violent_cases} violent).")
    else:
        print("Unknown Status")


Available Locations:
- Nairobi: Kilimani, Kibera, Westlands, Eastleigh
- Mombasa: Nyali, Likoni, Kisauni, Tudor
- Kiambu: Ruiru, Thika, Kikuyu, Limuru
- Nakuru: Naivasha, Gilgil, Molo, Rongai

Available Crimes:
Murder, Robbery with Violence, Burglary, Assault, Petty Theft, Fraud, Pickpocketing, GBV, Theft
----------------------------------------



Analysis Results for: NAIROBI
----------------------------------------
Top Crimes Reported:
* Petty Theft: 300 cases
* Pickpocketing: 210 cases
* Fraud: 159 cases
* Robbery with Violence: 153 cases
* Burglary: 120 cases

Overall Status for Search:
ORANGE ZONE / LOWER SEVERITY
Mostly property/petty records (789 petty vs 341 violent).


In [10]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# load the dataset
data = pd.read_csv('kenya_county_crime.csv')

# clean column names just in case
data.columns = data.columns.str.strip()

def check_severity(crime):
    crime = str(crime).lower()
    
    violent_crimes = ['murder', 'robbery', 'gbv', 'assault']
    petty_crimes = ['theft', 'burglary', 'pickpocketing', 'fraud']
    
    for c in violent_crimes:
        if c in crime:
            return 'Violent'
            
    for c in petty_crimes:
        if c in crime:
            return 'Petty'
            
    return 'Other'

# function to determine zone status for a specific location
def get_location_zone(county, ward):
    # get all crimes for this specific ward
    loc_data = data[(data['County'] == county) & (data['Ward'] == ward)]
    severity_totals = loc_data.groupby('Severity')['Cases'].sum()
    
    violent_cases = severity_totals.get('Violent', 0)
    petty_cases = severity_totals.get('Petty', 0)
    
    if violent_cases >= petty_cases and violent_cases > 0:
        return "RED ZONE"
    elif petty_cases > violent_cases:
        return "ORANGE ZONE"
    else:
        return "UNKNOWN"

# add severity column
data['Severity'] = data['Crime_Type'].apply(check_severity)

print("Available Locations:")
counties = data['County'].dropna().unique()

for c in counties:
    wards = data[data['County'] == c]['Ward'].unique()
    ward_list = ", ".join([str(w).strip() for w in wards])
    print(f"- {c}: {ward_list}")

print("\nAvailable Crimes:")
crimes = data['Crime_Type'].dropna().unique()
crime_list = ", ".join([str(cr).strip() for cr in crimes])
print(crime_list)

print("-" * 40)

# get user input
search_term = input("Enter a county, ward, or crime to analyze: ").strip()

# filter by county, ward, OR crime type
mask = data['County'].str.contains(search_term, case=False, na=False) | \
       data['Ward'].str.contains(search_term, case=False, na=False) | \
       data['Crime_Type'].str.contains(search_term, case=False, na=False)

filtered_data = data[mask]

if len(filtered_data) == 0:
    print("No data found for that search. Please try again.")
else:
    print(f"\nAnalysis Results for: {search_term.upper()}")
    print("-" * 40)
    
    # check if the user searched for a crime instead of a location
    is_crime_search = any(data['Crime_Type'].str.contains(search_term, case=False, na=False))
    
    if is_crime_search:
        print("Top Locations for this Crime:")
        # group by County and Ward
        top_locations = filtered_data.groupby(['County', 'Ward'])['Cases'].sum().sort_values(ascending=False)
        
        for loc, count in top_locations.head(5).items():
            county_name = loc[0]
            ward_name = loc[1]
            
            # evaluate the zone for this specific location
            zone_status = get_location_zone(county_name, ward_name)
            
            print(f"* {county_name} ({ward_name}): {count} cases [{zone_status}]")
            
    else:
        print("Top Crimes Reported:")
        top_crimes = filtered_data.groupby('Crime_Type')['Cases'].sum().sort_values(ascending=False)
        for crime, count in top_crimes.head(5).items():
            print(f"* {crime}: {count} cases")

        # overall severity check ONLY if searching by location
        severity_totals = filtered_data.groupby('Severity')['Cases'].sum()
        violent_cases = severity_totals.get('Violent', 0)
        petty_cases = severity_totals.get('Petty', 0)

        print("\nOverall Status for Search:")
        if violent_cases >= petty_cases and violent_cases > 0:
            print("RED ZONE / HIGH SEVERITY")
            print(f"Warning: High number of violent records ({violent_cases} violent vs {petty_cases} petty).")
        elif petty_cases > violent_cases:
            print("ORANGE ZONE / LOWER SEVERITY")
            print(f"Mostly property/petty records ({petty_cases} petty vs {violent_cases} violent).")
        else:
            print("Unknown Status")


Available Locations:
- Nairobi: Kilimani, Kibera, Westlands, Eastleigh
- Mombasa: Nyali, Likoni, Kisauni, Tudor
- Kiambu: Ruiru, Thika, Kikuyu, Limuru
- Nakuru: Naivasha, Gilgil, Molo, Rongai

Available Crimes:
Murder, Robbery with Violence, Burglary, Assault, Petty Theft, Fraud, Pickpocketing, GBV, Theft
----------------------------------------



Analysis Results for: NAIROBI
----------------------------------------
Top Crimes Reported:
* Petty Theft: 300 cases
* Pickpocketing: 210 cases
* Fraud: 159 cases
* Robbery with Violence: 153 cases
* Burglary: 120 cases

Overall Status for Search:
ORANGE ZONE / LOWER SEVERITY
Mostly property/petty records (789 petty vs 341 violent).
